In [7]:
import os
from dotenv import load_dotenv 
load_dotenv()

True

In [8]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WebBaseLoader


In [11]:
urls = [
    "https://en.wikipedia.org/wiki/Bhagat_Singh",
    "https://en.wikipedia.org/wiki/Bhagat_Singh#Early_life",
    "https://en.wikipedia.org/wiki/Bhagat_Singh#Influence"
]

docs = []

for url in urls:
    docs.extend(WebBaseLoader(url).load())

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings()

vectorestore = FAISS.from_documents(
    chunks,
    embedding
)

retriever = vectorestore.as_retriever()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [13]:
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(
    retriever,
    "retriever_vectore_db_blog",
    "search and run information about bhagat singh"
)
retriever_tool

StructuredTool(name='retriever_vectore_db_blog', description='search and run information about bhagat singh', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000214B64F2F20>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000214B64F2E80>)

In [8]:
urls2 = (
    "https://en.wikipedia.org/wiki/Mahatma_Gandhi#Three_years_in_London"
)

doc2 = WebBaseLoader(urls2).load()
splitter2 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks2 = splitter.split_documents(doc2)

embedding2 = HuggingFaceEmbeddings()

vectorestore = FAISS.from_documents(
    chunks2,
    embedding2
)

retriever2 = vectorestore.as_retriever()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [15]:
from langchain_core.tools import create_retriever_tool
retriever_tool_mahatma = create_retriever_tool(
    retriever2,
    "retriever_mahatma blog",
    "search and run information about mhatama gandhi"
)
retriever_tool_mahatma

StructuredTool(name='retriever_mahatma blog', description='search and run information about mhatama gandhi', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000214E430F380>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000214E430FE20>)

In [17]:
tools = [retriever_tool,retriever_tool_mahatma]
tools

[StructuredTool(name='retriever_vectore_db_blog', description='search and run information about bhagat singh', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000214B64F2F20>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000214B64F2E80>),
 StructuredTool(name='retriever_mahatma blog', description='search and run information about mhatama gandhi', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000214E430F380>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000214E430FE20>)]

In [18]:
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]
    

In [19]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-120b"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000214E35D46E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000214E35D5940>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
def agent(state):
    print("----- Call Agent ------")

    messages = state["messages"]

    model = ChatGroq(
        model="openai/gpt-oss-120b"
    )

    model = model.bind_tools(tools)

    response = model.invoke(messages)

    return {
        "messages": [response]
    }

In [2]:
from typing import Literal
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel , Field
def grade_document(state)->Literal["generate","rewrite"]:
    """detrmine whether the retrieved documents are relevant to the question 
    
    Args(messages): the current state
    
    Returns :
        str:A descision for whether the documnet is relevant or not 
    """
    print("----check Relavance----")
    class Grade(BaseModel):
        """ Binary Score for relevance Check. """
        
        binary_score:str=Field(description="Relevance Score 'Yes' or 'No' ")
    model = ChatGroq("openai/gpt-oss-120b")
    llm_with_tool = model.with_structured_output(Grade)   
    
    prompt = PromptTemplate(
        template="""
        
        
        """
    )
    
    chain = prompt | llm_with_tool
    messages = state["messages"]
    last_message = messages[-1]
    question = messages[0].content
    docs = last_message.content
    scored_resul = chain.invoke({
        "question":question,
        "context":docs
    })
    score = scored_resul.binary_score
    
    if score == "yes":
        print("--Descision:Docs Relevant")
        return "generate"
    else:
        print("descision:Docs Not Relevant--")
        print(score)
        return "rewrite"

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

KeyboardInterrupt: 